### SQL Databases

In [1]:
## create SQlite Database 
import sqlite3
import os 

os.makedirs("data/databases" , exist_ok=True)

In [2]:
## create ample database
conn = sqlite3.connect("data/databases/company.db")
cursor = conn.cursor()

In [6]:
## create Table 
cursor.execute('''CREATE TABLE IF NOT EXISTS employees
               (id INTEGER PRIMARY KEY , 
                name TEXT , 
                role TEXT , 
                department TEXT , 
                salary REAL  )''')



In [7]:
## create Table 
cursor.execute('''CREATE TABLE IF NOT EXISTS projects
               (id INTEGER PRIMARY KEY , 
                name TEXT , 
                status TEXT , 
                budget REAL , 
                lead_id INTEGER )''')



In [20]:
employees = [
    (1, 'Alice Smith', 'Junior developer', 'Designer', 75000),
    (2, 'Michael Brown', 'Backend developer', 'Engineer', 88000),
    (3, 'Sarah Wilson', 'Project manager', 'Manager', 105000),
    (4, 'David Lee', 'Frontend developer', 'Engineer', 82000)
]

projects = [
    (1,'E-commerce platform', 'Active', 200000, 1),
    (2,'Mobile banking app', 'Completed', 350000, 2),
    (3, 'Healthcare dashboard', 'Pending', 180000, 3),
    (4 ,'AI chatbot system', 'Active', 250000, 2)
]

In [22]:
cursor.executemany('INSERT OR REPLACE INTO employees VALUES (?,?,?,?,?)' , employees)
cursor.executemany('INSERT OR REPLACE INTO projects VALUES (?,?,?,?,?)' , projects)

In [23]:
cursor.execute('SELECT * from employees')


In [24]:
conn.commit()
conn.close()

### Database Content Extration

In [25]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

/Users/macbookpro/Documents/Projects/Rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
## Method 1 SQL database utility 
db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")

## Get database info 
print(f"Tables : {db.get_usable_table_names()}")
print(f"\n Table DDl :")
print(db.get_table_info())

Tables : ['employees', 'products', 'projects']

 Table DDl :

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	Alice Smith	Junior developer	Designer	75000.0
2	Michael Brown	Backend developer	Engineer	88000.0
3	Sarah Wilson	Project manager	Manager	105000.0
*/


CREATE TABLE products (
	id INTEGER, 
	name TEXT, 
	category TEXT, 
	description TEXT, 
	price REAL, 
	stock INTEGER, 
	specifications TEXT, 
	tags TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from products table:
id	name	category	description	price	stock	specifications	tags
1	ProBook Laptop	Electronics	High-performance laptop with 16GB RAM and 512GB SSD.	1200.0	15	{"processor": "Intel Core i7", "ram": "16GB", "storage": "512GB SSD"}	["laptop", "computer", "electronics"]
2	Wireless Mouse	Accessories	Ergonomic wireless mouse designed for everyday productivity.	35.0	50	{"connection": "Bluetooth", "battery"

In [1]:
import sqlite3
from typing import List
from langchain_core.documents import Document

## Method 2 Custom SQL to document conversion 
print("Custom SQL Processing")

def sql_to_documents(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    documents = []

    ## Strategy 1 create document for each table 
    cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]

        ## get table schema 
        cursor.execute(f"PRAGMA table_info({table_name}) ;")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]          

        ## get table data 
        cursor.execute(f"SELECT * FROM {table_name} ")
        rows = cursor.fetchall()

        ## create table overview document 
        table_content = f"Table : {table_name}\n"           
        table_content += f"Columns : {', '.join(column_names)}\n"
        table_content += f"Total records: {len(rows)}\n\n"

        ## add sample record 
        table_content += "Sample Records :\n"
        for row in rows[:5]:
            record = dict(zip(column_names, row))           # <- fixed (row)
            table_content += f"{record}\n"

        doc = Document(
            page_content=table_content,
            metadata={                                      # <- fixed (spelling)
                'source': db_path,
                'table_name': table_name,
                'num_records': len(rows),
                'data_type': 'sql_table'
            }
        )
        documents.append(doc)

    ## Strategy 2 : create relationship documents 
    ## Example : join employees and projects 
    cursor.execute("""
        SELECT e.name , e.role , p.name as project_name , p.status 
        FROM employees e 
        JOIN projects p ON e.id = p.lead_id
    """)

    relationships = cursor.fetchall()
    rel_content = "Employee-Project Relationships:\n\n"
    for rel in relationships:
        rel_content += f"{rel[0]} ({rel[1]}) leads {rel[2]} - Status: {rel[3]}\n"

    rel_doc = Document(
        page_content=rel_content,
        metadata={
            'source': db_path,
            'data_type': 'sql_relationships',
            'query': 'employee_project_join'
        }
    )
    documents.append(rel_doc)

    conn.close()
    return documents                                        # <- fixed


Custom SQL Processing


In [2]:
sql_to_documents("data/databases/company.db")

[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'products', 'num_records': 5, 'data_type': 'sql_table'}, page_content='Table : products\nColumns : id, name, category, description, price, stock, specifications, tags\nTotal records: 5\n\nSample Records :\n{\'id\': 1, \'name\': \'ProBook Laptop\', \'category\': \'Electronics\', \'description\': \'High-performance laptop with 16GB RAM and 512GB SSD.\', \'price\': 1200.0, \'stock\': 15, \'specifications\': \'{"processor": "Intel Core i7", "ram": "16GB", "storage": "512GB SSD"}\', \'tags\': \'["laptop", "computer", "electronics"]\'}\n{\'id\': 2, \'name\': \'Wireless Mouse\', \'category\': \'Accessories\', \'description\': \'Ergonomic wireless mouse designed for everyday productivity.\', \'price\': 35.0, \'stock\': 50, \'specifications\': \'{"connection": "Bluetooth", "battery": "AA Battery", "range": "10 meters"}\', \'tags\': \'["mouse", "wireless", "accessories"]\'}\n{\'id\': 3, \'name\': \'Mechanical Keyboard\', \'